# DMP Bridge — Batch PDF Pipeline Test

This notebook runs all PDFs in `data/raw_pdfs/` through:

```text
PDF
↓
pdfplumber extraction
↓
rule-based structure detection
↓
DMPTool narrative JSON builder
↓
debug CSV + final JSON outputs
```


## Part 1 — Imports


In [1]:
from pathlib import Path
import pandas as pd
import json

from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure
from dmpbridge.processing.structure_json_builder import save_narrative_json



## Part 2 — Project paths


In [2]:
from pathlib import Path

cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

raw_pdf_dir = project_root / "data" / "raw_pdfs"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

pdfplumber_output_dir = project_root / "data" / "pdfplumber_blocks"
extracted_text_dir = project_root / "data" / "extracted_text"
markdown_dir = project_root / "data" / "markdown"
debug_output_dir = project_root / "outputs" / "debug"
structure_json_dir = project_root / "data" / "structure_json"

for path in [
    pdfplumber_output_dir,
    extracted_text_dir,
    markdown_dir,
    debug_output_dir,
    structure_json_dir,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw PDF directory exists:", raw_pdf_dir.exists())
print("Skeleton exists:", skeleton_path.exists())
print("Markdown directory exists:", markdown_dir.exists())

Project root: c:\Users\Nahid\dmpbridge
Raw PDF directory exists: True
Skeleton exists: True
Markdown directory exists: True


## Part 3 — Find all PDFs


In [3]:
pdf_paths = sorted(raw_pdf_dir.glob("*.pdf"))

print("Number of PDFs found:", len(pdf_paths))

for pdf_path in pdf_paths:
    print("-", pdf_path.name)


Number of PDFs found: 10
- sample1.pdf
- sample10.pdf
- sample2.pdf
- sample3.pdf
- sample4.pdf
- sample5.pdf
- sample6.pdf
- sample7.pdf
- sample8.pdf
- sample9.pdf


In [4]:
# Change this index if you want to test a different PDF first.
sample_index = 0

pdf_path = pdf_paths[sample_index]

print("Testing:", pdf_path.name)

blocks = save_pdfplumber_outputs(pdf_path)
structured_blocks = detect_structure(blocks)

df = pd.DataFrame(structured_blocks)

debug_columns = [
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "label"
]

# document_format existed in an older detector version.
# Keep it only if your current detector outputs it.
if "document_format" in df.columns:
    debug_columns.append("document_format")

display(df[debug_columns].head(80))

print(df["label"].value_counts())


Testing: sample1.pdf
[2026-05-20 13:03:52] Extracting line-level text with pdfplumber: sample1.pdf
[2026-05-20 13:03:52] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample1.json
[2026-05-20 13:03:52] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample1.txt
[2026-05-20 13:03:52] Saved Markdown text: C:\Users\Nahid\dmpbridge\data\markdown\sample1.md


,page,line_order,text,avg_font_size,is_bold,label
0,1,1,DATA MANAGEMENT AND SHARING PLAN,11.04,True,document_title
1,1,2,Element 1: Data Type:,11.04,True,section
2,1,3,A. Types and amount of scientific data expecte...,11.04,True,subsection
3,1,4,This secondary data analysis project will anal...,11.04,False,content
4,1,5,and the publicly available NHANES cohorts (wri...,11.04,False,content
...,...,...,...,...,...,...
74,2,36,"PI, Trial & IRB Coordinator, and Statistician/...",11.04,False,content
75,2,37,"ensure that the datasets, protocols, and codes...",11.04,False,content
76,2,38,the main outcome manuscript has been published...,11.04,False,content
77,2,39,University of California San Diego Library rep...,11.04,False,content


label
content           64
subsection         8
section            6
document_title     1
Name: count, dtype: int64


## Part 5 — Save one sample debug CSV and narrative JSON


In [5]:
pdf_stem = pdf_path.stem

csv_output_path = debug_output_dir / f"{pdf_stem}_structured_lines.csv"
final_json_path = structure_json_dir / f"{pdf_stem}_pdfplumber.json"

# Re-create df if this cell is run independently after structured_blocks exists.
df = pd.DataFrame(structured_blocks)

debug_columns = [
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "label"
]

if "document_format" in df.columns:
    debug_columns.append("document_format")

df[debug_columns].to_csv(
    csv_output_path,
    index=False,
    encoding="utf-8"
)

final_json = save_narrative_json(
    structured_blocks=structured_blocks,
    output_path=final_json_path,
    skeleton_path=skeleton_path
)

sections = final_json["narrative"]["template"]["section"]

print("Saved CSV:", csv_output_path)
print("Saved JSON:", final_json_path)
print("Template title:", final_json["narrative"]["template"]["title"])
print("Number of sections:", len(sections))

for sec in sections:
    print(sec["order"], sec["title"], "| questions:", len(sec["question"]))


[2026-05-20 13:03:52] Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample1_pdfplumber.json
Saved CSV: c:\Users\Nahid\dmpbridge\outputs\debug\sample1_structured_lines.csv
Saved JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample1_pdfplumber.json
Template title: DATA MANAGEMENT AND SHARING PLAN
Number of sections: 6
1 Element 1: Data Type: | questions: 3
2 Element 2: Related Tools, Software and/or Code: | questions: 1
3 Element 3: Standards: | questions: 1
4 Element 4: Data Preservation, Access, and Associated Timelines: | questions: 3
5 Element 5: Access, Distribution, or Reuse Considerations: | questions: 2
6 Element 6: Oversight of Data Management and Sharing: | questions: 1


## Part 6 — Run all PDFs


In [6]:
results = []

for pdf_path in pdf_paths:
    print("\n" + "=" * 80)
    print("Processing:", pdf_path.name)

    try:
        blocks = save_pdfplumber_outputs(pdf_path)
        structured_blocks = detect_structure(blocks)
        df = pd.DataFrame(structured_blocks)

        pdf_stem = pdf_path.stem

        csv_output_path = debug_output_dir / f"{pdf_stem}_structured_lines.csv"
        final_json_path = structure_json_dir / f"{pdf_stem}_pdfplumber.json"

        debug_columns = [
            "page",
            "line_order",
            "text",
            "avg_font_size",
            "is_bold",
            "label"
        ]

        if "document_format" in df.columns:
            debug_columns.append("document_format")

        if not df.empty:
            df[debug_columns].to_csv(
                csv_output_path,
                index=False,
                encoding="utf-8"
            )
        else:
            pd.DataFrame(columns=debug_columns).to_csv(
                csv_output_path,
                index=False,
                encoding="utf-8"
            )

        final_json = save_narrative_json(
            structured_blocks=structured_blocks,
            output_path=final_json_path,
            skeleton_path=skeleton_path
        )

        sections = final_json["narrative"]["template"]["section"]
        template_title = final_json["narrative"]["template"]["title"]

        question_count = sum(
            len(sec.get("question", []))
            for sec in sections
        )

        result = {
            "pdf": pdf_path.name,
            "status": "success",
            "lines": len(blocks),
            "structured_blocks": len(structured_blocks),
            "template_title": template_title,
            "sections": len(sections),
            "questions": question_count,
            "csv_output": str(csv_output_path),
            "json_output": str(final_json_path),
            "error": None
        }

        if "document_format" in df.columns and not df.empty:
            result["document_format"] = df["document_format"].iloc[0]
        else:
            result["document_format"] = None

        print("Sections:", len(sections))
        print("Questions:", question_count)
        print("Saved:", final_json_path)

    except Exception as e:
        result = {
            "pdf": pdf_path.name,
            "status": "failed",
            "lines": None,
            "structured_blocks": None,
            "document_format": None,
            "template_title": None,
            "sections": None,
            "questions": None,
            "csv_output": None,
            "json_output": None,
            "error": str(e)
        }

        print("FAILED:", e)

    results.append(result)

summary_df = pd.DataFrame(results)
summary_df



Processing: sample1.pdf
[2026-05-20 13:03:52] Extracting line-level text with pdfplumber: sample1.pdf
[2026-05-20 13:03:52] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample1.json
[2026-05-20 13:03:52] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample1.txt
[2026-05-20 13:03:52] Saved Markdown text: C:\Users\Nahid\dmpbridge\data\markdown\sample1.md
[2026-05-20 13:03:52] Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample1_pdfplumber.json
Sections: 6
Questions: 11
Saved: c:\Users\Nahid\dmpbridge\data\structure_json\sample1_pdfplumber.json

Processing: sample10.pdf
[2026-05-20 13:03:52] Extracting line-level text with pdfplumber: sample10.pdf
[2026-05-20 13:03:52] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample10.json
[2026-05-20 13:03:52] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample10.txt
[2026-05-20 13:03:52] Saved Markdown text: C:\Users\Nahid\dmpbridge\

,pdf,status,lines,structured_blocks,template_title,sections,questions,csv_output,json_output,error,document_format
0,sample1.pdf,success,79,79,DATA MANAGEMENT AND SHARING PLAN,6,11,c:\Users\Nahid\dmpbridge\outputs\debug\sample1...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
1,sample10.pdf,success,68,68,DATA MANAGEMENT,6,6,c:\Users\Nahid\dmpbridge\outputs\debug\sample1...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
2,sample2.pdf,success,171,171,Center for Bio-Inspired Energy Science,4,4,c:\Users\Nahid\dmpbridge\outputs\debug\sample2...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
3,sample3.pdf,success,69,69,CPS 2015,5,4,c:\Users\Nahid\dmpbridge\outputs\debug\sample3...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
4,sample4.pdf,success,185,185,Data and Computing Resource Management Plan,5,5,c:\Users\Nahid\dmpbridge\outputs\debug\sample4...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
5,sample5.pdf,success,81,81,DATA MANAGEMENT PLAN,6,6,c:\Users\Nahid\dmpbridge\outputs\debug\sample5...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
6,sample6.pdf,success,24,24,Data Management Plan:,5,5,c:\Users\Nahid\dmpbridge\outputs\debug\sample6...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
7,sample7.pdf,success,17,17,Resource/Data Sharing Plan,1,1,c:\Users\Nahid\dmpbridge\outputs\debug\sample7...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
8,sample8.pdf,success,59,59,"Univ. of California, Riverside Data Management...",6,6,c:\Users\Nahid\dmpbridge\outputs\debug\sample8...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
9,sample9.pdf,success,85,85,CAREER: HIGH-RESOLUTION NMR FOR PARAMAGNETIC S...,6,6,c:\Users\Nahid\dmpbridge\outputs\debug\sample9...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None


## Part 7 — Save batch summary


In [7]:
summary_path = debug_output_dir / "pdfplumber_batch_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8")

print("Saved batch summary:", summary_path)
display(summary_df)


Saved batch summary: c:\Users\Nahid\dmpbridge\outputs\debug\pdfplumber_batch_summary.csv


,pdf,status,lines,structured_blocks,template_title,sections,questions,csv_output,json_output,error,document_format
0,sample1.pdf,success,79,79,DATA MANAGEMENT AND SHARING PLAN,6,11,c:\Users\Nahid\dmpbridge\outputs\debug\sample1...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
1,sample10.pdf,success,68,68,DATA MANAGEMENT,6,6,c:\Users\Nahid\dmpbridge\outputs\debug\sample1...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
2,sample2.pdf,success,171,171,Center for Bio-Inspired Energy Science,4,4,c:\Users\Nahid\dmpbridge\outputs\debug\sample2...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
3,sample3.pdf,success,69,69,CPS 2015,5,4,c:\Users\Nahid\dmpbridge\outputs\debug\sample3...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
4,sample4.pdf,success,185,185,Data and Computing Resource Management Plan,5,5,c:\Users\Nahid\dmpbridge\outputs\debug\sample4...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
5,sample5.pdf,success,81,81,DATA MANAGEMENT PLAN,6,6,c:\Users\Nahid\dmpbridge\outputs\debug\sample5...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
6,sample6.pdf,success,24,24,Data Management Plan:,5,5,c:\Users\Nahid\dmpbridge\outputs\debug\sample6...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
7,sample7.pdf,success,17,17,Resource/Data Sharing Plan,1,1,c:\Users\Nahid\dmpbridge\outputs\debug\sample7...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
8,sample8.pdf,success,59,59,"Univ. of California, Riverside Data Management...",6,6,c:\Users\Nahid\dmpbridge\outputs\debug\sample8...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None
9,sample9.pdf,success,85,85,CAREER: HIGH-RESOLUTION NMR FOR PARAMAGNETIC S...,6,6,c:\Users\Nahid\dmpbridge\outputs\debug\sample9...,c:\Users\Nahid\dmpbridge\data\structure_json\s...,None,None


## Part 8 — Quick quality checks


In [8]:
# Check for common problems:
# 1. Missing template title
# 2. Zero sections
# 3. Suspiciously many sections
# 4. Failed files

quality_checks = summary_df.copy()

quality_checks["missing_title"] = quality_checks["template_title"].isna()
quality_checks["zero_sections"] = quality_checks["sections"].fillna(0).eq(0)
quality_checks["many_sections"] = quality_checks["sections"].fillna(0).gt(12)
quality_checks["failed"] = quality_checks["status"].ne("success")

display(quality_checks[[
    "pdf",
    "status",
    "template_title",
    "sections",
    "questions",
    "missing_title",
    "zero_sections",
    "many_sections",
    "failed",
    "error"
]])


,pdf,status,template_title,sections,questions,missing_title,zero_sections,many_sections,failed,error
0,sample1.pdf,success,DATA MANAGEMENT AND SHARING PLAN,6,11,False,False,False,False,None
1,sample10.pdf,success,DATA MANAGEMENT,6,6,False,False,False,False,None
2,sample2.pdf,success,Center for Bio-Inspired Energy Science,4,4,False,False,False,False,None
3,sample3.pdf,success,CPS 2015,5,4,False,False,False,False,None
4,sample4.pdf,success,Data and Computing Resource Management Plan,5,5,False,False,False,False,None
5,sample5.pdf,success,DATA MANAGEMENT PLAN,6,6,False,False,False,False,None
6,sample6.pdf,success,Data Management Plan:,5,5,False,False,False,False,None
7,sample7.pdf,success,Resource/Data Sharing Plan,1,1,False,False,False,False,None
8,sample8.pdf,success,"Univ. of California, Riverside Data Management...",6,6,False,False,False,False,None
9,sample9.pdf,success,CAREER: HIGH-RESOLUTION NMR FOR PARAMAGNETIC S...,6,6,False,False,False,False,None


## Part 9 — Inspect any generated JSON


In [9]:
# Change this to inspect a different output.
inspect_pdf_stem = pdf_paths[0].stem
inspect_json_path = structure_json_dir / f"{inspect_pdf_stem}_pdfplumber.json"

with open(inspect_json_path, "r", encoding="utf-8") as f:
    generated_json = json.load(f)

sections = generated_json["narrative"]["template"]["section"]

print("Inspecting:", inspect_json_path)
print("Template title:", generated_json["narrative"]["template"]["title"])
print("Sections:", len(sections))

for sec in sections:
    print(sec["id"], sec["title"], "| questions:", len(sec["question"]))


Inspecting: c:\Users\Nahid\dmpbridge\data\structure_json\sample1_pdfplumber.json
Template title: DATA MANAGEMENT AND SHARING PLAN
Sections: 6
1 Element 1: Data Type: | questions: 3
2 Element 2: Related Tools, Software and/or Code: | questions: 1
3 Element 3: Standards: | questions: 1
4 Element 4: Data Preservation, Access, and Associated Timelines: | questions: 3
5 Element 5: Access, Distribution, or Reuse Considerations: | questions: 2
6 Element 6: Oversight of Data Management and Sharing: | questions: 1


## Part 10 — Inspect first section


In [10]:
sections[0] if sections else None


{'id': 1,
 'title': 'Element 1: Data Type:',
 'description': None,
 'order': 1,
 'question': [{'id': 1,
   'text': 'A. Types and amount of scientific data expected to be generated in the project:',
   'order': 1,
   'answer': {'id': 1,
    'json': {'type': 'textArea',
     'meta': {'schemaVersion': '1.0'},
     'answer': 'This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies\nand the publicly available NHANES cohorts (wrist NHANES 2011-2014; hip/counts-NHANES 2003-2006).\nThe studies include (i) the RISE Study, (ii) the SOL-VIDA Study, (iii) the iWATCH Study, (iv) the MOCA Study,\n(v) the PHASE Study, (vi) the AusDiab Study, (vii) the ACT Study, and (viii) the WHISH accelerometer sub-\nstudy.'}}},
  {'id': 2,
   'text': 'B. Scientific data that will be preserved and shared, and the rationale for doing so:',
   'order': 2,
   'answer': {'id': 2,
    'json': {'type': 'textArea',
     'meta': {'schemaVersion': '1.0'},
     'answer'